In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
portfolio = pd.read_csv('../data/portfolio.csv')
profile = pd.read_csv('../data/profile.csv')
transcript = pd.read_csv('../data/transcript.csv')

---

# profile.csv

## 1. 결측치 확인

- `gender`, `income`, `age` 컬럼의 결측치 수가 2175로 동일하여,<br>해당 결측치가 동일한 행에서 발생한 것인지 확인 

In [3]:
# 1. 조건 정의
cond_gender = profile['gender'].isnull()
cond_income = profile['income'].isnull()
cond_age = profile['age'] == 118

# 2. id 집합 생성
set_gender = set(profile.loc[cond_gender, 'id'])
set_income = set(profile.loc[cond_income, 'id'])
set_age = set(profile.loc[cond_age, 'id'])

# 3. 각각 개수 확인
print("gender 결측 수:", len(set_gender))
print("income 결측 수:", len(set_income))
print("age == 118 수:", len(set_age))
print("-" * 40)

# 4. 집합 비교 (완전 동일 여부)
print("gender == income:", set_gender == set_income)
print("gender == age:", set_gender == set_age)
print("income == age:", set_income == set_age)
print("-" * 40)

# 5. 교집합 확인
print("gender ∩ income:", len(set_gender & set_income))
print("gender ∩ age:", len(set_gender & set_age))
print("income ∩ age:", len(set_income & set_age))
print("-" * 40)

# 6. 차이 확인 (어디가 다른지)
print("gender - income:", len(set_gender - set_income))
print("income - gender:", len(set_income - set_gender))
print("gender - age:", len(set_gender - set_age))
print("age - gender:", len(set_age - set_gender))
print("-" * 40)

# 7. 완전 동일 패턴 여부 (3개 다 동일)
all_equal = (set_gender == set_income == set_age)
print("세 조건 모두 동일한 id 집합인지:", all_equal)

gender 결측 수: 2175
income 결측 수: 2175
age == 118 수: 2175
----------------------------------------
gender == income: True
gender == age: True
income == age: True
----------------------------------------
gender ∩ income: 2175
gender ∩ age: 2175
income ∩ age: 2175
----------------------------------------
gender - income: 0
income - gender: 0
gender - age: 0
age - gender: 0
----------------------------------------
세 조건 모두 동일한 id 집합인지: True


> 같은 행에서 발생함을 확인

## 2. gender 컬럼의 other

- `gender` 컬럼에서 `other` 값을 어떻게 처리할까

In [4]:
profile['gender'].value_counts(dropna=False, normalize=True).round(3)

gender
M      0.499
F      0.361
NaN    0.128
O      0.012
Name: proportion, dtype: float64

> `other`의 비율은 약 1%로 버려도 괜찮을 것 같음
<br> NaN 값을 어떻게 처리할지가 더 문제ㅠ

In [5]:
# 데이터 타입 date형식으로 변환
profile["became_member_on"] = pd.to_datetime(profile["became_member_on"], format="%Y%m%d")


# channels마다 파생변수 생성
portfolio['web'] = portfolio['channels'].astype(str).str.contains('web').astype(int)
portfolio['email'] = portfolio['channels'].astype(str).str.contains('email').astype(int)
portfolio['mobile'] = portfolio['channels'].astype(str).str.contains('mobile').astype(int)
portfolio['social'] = portfolio['channels'].astype(str).str.contains('social').astype(int)

# 기존 channels 컬럼 제거
portfolio = portfolio.drop('channels', axis=1)

In [6]:
# 딕셔너리처럼 생긴 문자열을 진짜 딕셔너리로 변환
transcript['value'] = transcript['value'].apply(ast.literal_eval)

# 딕셔너리의 키 -> 새로운 컬럼
value_df = pd.DataFrame(transcript['value'].tolist())
transcript = pd.concat([transcript, value_df], axis=1)

# offer id를 offer_id로 컬럼명 통일
transcript['offer_id'] = transcript['offer_id'].fillna(transcript['offer id'])

# offer id 컬럼 제거
transcript = transcript.drop('offer id', axis=1)

# value 컬럼 제거
transcript = transcript.drop('value', axis=1)

In [7]:
# profile의 필요없는 Unnamed:0 컬럼 제거
profile = profile.drop('Unnamed: 0', axis=1)

# transcript 기준으로 profile 데이터를 Left Join
merged_df = pd.merge(transcript, profile, left_on='person', right_on='id', how='left')

# 필요 없는 id 컬럼(person과 중복)은 버리기
merged_df = merged_df.drop(columns='id')

In [8]:
# 결측치 처리

# gender의 결측치 'Unknown'으로 채우기 
merged_df['gender'] = merged_df['gender'].fillna('Unknown')


# age의 118을 결측치(NaN)로 바꿔주기 
merged_df['age'] = merged_df['age'].replace(118, np.nan)
# income은 이미 결측치(NaN) 상태

In [9]:
# portfolio 테이블도 병합

# portfolio 테이블의 필요없는 인덱스 컬럼 제거
portfolio = portfolio.drop('Unnamed: 0', axis=1)

all_merge_df = pd.merge(
    merged_df,
    portfolio,
    left_on='offer_id',
    right_on='id',
    how='left'
)

all_merge_df = all_merge_df.drop(columns='id')

# reward 컬럼명 변경(명확하게)
all_merge_df = all_merge_df.rename(columns={
    "reward_x": "transcript_reward",
    "reward_y": "portfolio_reward"
})

In [10]:
# offer_id 이름 변경 (쿠폰명_difficulty_reward_duration)
portfolio['offer_name'] = (
    portfolio['offer_type'] + '_' + 
    portfolio['difficulty'].astype(str) + '_' + 
    portfolio['reward'].astype(str) + '_' + 
    portfolio['duration'].astype(str)
)
# id : key, offer_name : value
offer_name_dict = portfolio.set_index('id')['offer_name'].to_dict()
all_merge_df['offer_id'] = all_merge_df['offer_id'].map(offer_name_dict)


# 사람(person)별로 먼저 묶고, 그 안에서 시간(time) 순서대로 오름차순 정렬
all_merge_df = all_merge_df.sort_values(by=['person', 'time', 'Unnamed: 0']) # - Unnamed: 0 순서 추가

In [11]:
all_merge_df

,Unnamed: 0,person,event,time,amount,offer_id,transcript_reward,gender,age,became_member_on,income,portfolio_reward,difficulty,duration,offer_type,web,email,mobile,social
55972,55972,0009655768c64bdeb2e877511632db8f,offer received,168,NaN,informational_0_0_3,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,3.0,informational,0.0,1.0,1.0,1.0
77705,77705,0009655768c64bdeb2e877511632db8f,offer viewed,192,NaN,informational_0_0_3,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,3.0,informational,0.0,1.0,1.0,1.0
89291,89291,0009655768c64bdeb2e877511632db8f,transaction,228,22.16,NaN,NaN,M,33.0,2017-04-21,72000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113605,113605,0009655768c64bdeb2e877511632db8f,offer received,336,NaN,informational_0_0_4,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,4.0,informational,1.0,1.0,1.0,0.0
139992,139992,0009655768c64bdeb2e877511632db8f,offer viewed,372,NaN,informational_0_0_4,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,4.0,informational,1.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258361,258361,ffff82501cea40309d5fdd7edcca4a07,transaction,576,14.23,NaN,NaN,F,45.0,2016-11-25,62000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
258362,258362,ffff82501cea40309d5fdd7edcca4a07,offer completed,576,NaN,discount_10_2_7,2.0,F,45.0,2016-11-25,62000.0,2.0,10.0,7.0,discount,1.0,1.0,1.0,0.0
262475,262475,ffff82501cea40309d5fdd7edcca4a07,offer viewed,582,NaN,discount_10_2_7,NaN,F,45.0,2016-11-25,62000.0,2.0,10.0,7.0,discount,1.0,1.0,1.0,0.0
274809,274809,ffff82501cea40309d5fdd7edcca4a07,transaction,606,10.12,NaN,NaN,F,45.0,2016-11-25,62000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# 무엇을 위해 하는 코드인가? -> informational이 아닌 completed와 amount 경우만 선택하는 과정

# 조건 1: 쿠폰 타입이 bogo 이거나(in) discount 인 것
cond_offers = all_merge_df['offer_type'].isin(['bogo', 'discount'])

# 조건 2: 이벤트 종류가 transaction(결제) 인 것
cond_transactions = all_merge_df['event'] == 'transaction'

# 위 두 조건 중 하나라도 만족하는(|) 데이터만 쏙 뽑아서 덮어씌우기
target_df = all_merge_df[cond_offers | cond_transactions].copy()

# 잘 걸러졌는지 눈으로 확인해보기
print(target_df['offer_type'].value_counts(dropna=False))
print(target_df['event'].value_counts(dropna=False))
display(target_df.head())
display(target_df.shape)

offer_type
NaN         138953
bogo         71617
discount     69898
Name: count, dtype: int64
event
transaction        138953
offer received      61042
offer viewed        46894
offer completed     33579
Name: count, dtype: int64


,Unnamed: 0,person,event,time,amount,offer_id,transcript_reward,gender,age,became_member_on,income,portfolio_reward,difficulty,duration,offer_type,web,email,mobile,social
89291,89291,0009655768c64bdeb2e877511632db8f,transaction,228,22.16,NaN,NaN,M,33.0,2017-04-21,72000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
153401,153401,0009655768c64bdeb2e877511632db8f,offer received,408,NaN,bogo_5_5_5,NaN,M,33.0,2017-04-21,72000.0,5.0,5.0,5.0,bogo,1.0,1.0,1.0,1.0
168412,168412,0009655768c64bdeb2e877511632db8f,transaction,414,8.57,NaN,NaN,M,33.0,2017-04-21,72000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168413,168413,0009655768c64bdeb2e877511632db8f,offer completed,414,NaN,bogo_5_5_5,5.0,M,33.0,2017-04-21,72000.0,5.0,5.0,5.0,bogo,1.0,1.0,1.0,1.0
187554,187554,0009655768c64bdeb2e877511632db8f,offer viewed,456,NaN,bogo_5_5_5,NaN,M,33.0,2017-04-21,72000.0,5.0,5.0,5.0,bogo,1.0,1.0,1.0,1.0


(280468, 19)

In [13]:
# person당 offer_id를 하나의 행으로 설정하여, 흩어진 고객 행동의 순서를 보기 편하게 해주는 "피벗테이블 생성 코드"

# 1. 피벗을 돌릴 '쿠폰 이력서' 데이터만 빼내기
offers_df = target_df[target_df['event'] != 'transaction'].copy()

# 2. 안전한 금고에 보관할 '순수 영수증' 데이터만 빼내기
transactions_df = target_df[target_df['event'] == 'transaction'].copy()

print(f"피벗할 쿠폰 데이터: {len(offers_df)} 줄")
print(f"금고에 보관한 영수증: {len(transactions_df)} 줄")

피벗할 쿠폰 데이터: 141515 줄
금고에 보관한 영수증: 138953 줄


In [14]:
# 1. 시간 순서대로 예쁘게 줄 세우기 (시간이 꼬이면 안 되니까 필수)
offers_df = offers_df.sort_values(['person', 'offer_id', 'time'])

# 2. 'received' 이벤트가 등장할 때마다 1, 아니면 0인 깃발(Flag) 만들기
offers_df['is_received'] = (offers_df['event'] == 'offer received').astype(int)

# 3. [마법의 함수] 사람과 쿠폰 단위로 묶어서, 깃발을 누적해서 더하기 (Cumsum)
offers_df['offer_cycle'] = offers_df.groupby(['person', 'offer_id'])['is_received'].cumsum()

# 4. 이제 찝찝함 없이 피벗 돌리기 (기준점에 offer_cycle 추가!)
pivot_df = offers_df.pivot_table(
    index=['person', 'offer_id', 'offer_cycle'],  # "누구의 / 어떤 쿠폰의 / 몇 회차인가?"
    columns='event',
    values='time',
    aggfunc='min'  # 이제 한 회차 안에는 중복이 없으니 min을 써도 아무 왜곡이 안 일어납니다
).reset_index()

# 깔끔하게 정리
pivot_df.columns.name = None
pivot_df = pivot_df[['person', 'offer_id', 'offer_cycle', 'offer received', 'offer viewed', 'offer completed']]

display(pivot_df.head())
display(pivot_df.shape)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,1,408.0,456.0,414.0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,1,504.0,540.0,528.0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,1,576.0,NaN,576.0
3,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,1,168.0,216.0,NaN
4,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,2,576.0,630.0,NaN


(61042, 6)

In [15]:
# 1. 원본에서 offer_id와 offer_type 짝꿍 사전 만들기
offer_dict = offers_df[['offer_id', 'offer_type']].drop_duplicates().set_index('offer_id')['offer_type'].to_dict()

# 2. 피벗 테이블의 offer_id를 보고, 임시로 쿠폰 타입(bogo, discount)을 가져오기
temp_offer_type = pivot_df['offer_id'].map(offer_dict)

# 3. [핵심] 기존 숫자였던 'offer_cycle' 컬럼 위에 곧바로 덮어쓰기! 
pivot_df['offer_cycle'] = temp_offer_type.str.capitalize() + '_' + pivot_df['offer_cycle'].astype(str)


In [16]:
# 피벗테이블에 amount 붙이기

# 1. 금고(transactions_df)에서 영수증 알맹이만 꺼내기
transactions_df = transactions_df[['person', 'time', 'amount']]

# 2. 피벗 테이블(pivot_df)에 영수증(receipts) 1:1 도킹하기!
final_df = pivot_df.merge(
    transactions_df,
    left_on=['person', 'offer completed'],  # 왼쪽 표(피벗)의 도킹 기준: "누구의 / 언제 달성(completed)한 쿠폰인가?"
    right_on=['person', 'time'],      # 오른쪽 표(영수증)의 도킹 기준: "누가 / 언제(time) 결제했는가?"
    how='left'                        # 조인 방식: "피벗 표를 기준으로 하고, 영수증이 없으면 빈칸(NaN)으로 둬라!"
)

# 3. 도킹 끝나고 쓸모없어진 'time' 기둥 버리기
final_df = final_df.drop(columns=['time'])

# 4. 가슴이 웅장해지는 최종 결과물 확인!
display(final_df.head())
display(final_df.shape)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,amount
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,8.57
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,14.11
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,10.27
3,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_1,168.0,216.0,NaN,NaN
4,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_2,576.0,630.0,NaN,NaN


(61042, 7)

---

# transcript.csv

In [17]:
df_tran1 = transcript.sort_values(['person', 'time'])
df_tran1.head(30)

,Unnamed: 0,person,event,time,amount,offer_id,reward
55972,55972,0009655768c64bdeb2e877511632db8f,offer received,168,NaN,5a8bc65990b245e5a138643cd4eb9837,NaN
77705,77705,0009655768c64bdeb2e877511632db8f,offer viewed,192,NaN,5a8bc65990b245e5a138643cd4eb9837,NaN
89291,89291,0009655768c64bdeb2e877511632db8f,transaction,228,22.16,NaN,NaN
113605,113605,0009655768c64bdeb2e877511632db8f,offer received,336,NaN,3f207df678b143eea3cee63160fa8bed,NaN
139992,139992,0009655768c64bdeb2e877511632db8f,offer viewed,372,NaN,3f207df678b143eea3cee63160fa8bed,NaN
153401,153401,0009655768c64bdeb2e877511632db8f,offer received,408,NaN,f19421c1d4aa40978ebb69ca19b0e20d,NaN
168412,168412,0009655768c64bdeb2e877511632db8f,transaction,414,8.57,NaN,NaN
168413,168413,0009655768c64bdeb2e877511632db8f,offer completed,414,NaN,f19421c1d4aa40978ebb69ca19b0e20d,5.0
187554,187554,0009655768c64bdeb2e877511632db8f,offer viewed,456,NaN,f19421c1d4aa40978ebb69ca19b0e20d,NaN
204340,204340,0009655768c64bdeb2e877511632db8f,offer received,504,NaN,fafdcd668e3743c1bb461111dcafc2a4,NaN


In [18]:
transcript[(transcript['time'] == 0) & (transcript['event'] == 'offer completed')]

,Unnamed: 0,person,event,time,amount,offer_id,reward
12658,12658,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,offer completed,0,NaN,2906b810c7d4411798c6938adc9daaa5,2.0
12672,12672,fe97aa22dd3e48c8b143116a8403dd52,offer completed,0,NaN,fafdcd668e3743c1bb461111dcafc2a4,2.0
12679,12679,629fc02d56414d91bca360decdfa9288,offer completed,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,5.0
12692,12692,676506bad68e4161b9bbaffeb039626b,offer completed,0,NaN,ae264e3637204a6fb9bb56bc8210ddfd,10.0
12697,12697,8f7dd3b2afe14c078eb4f6e6fe4ba97d,offer completed,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,10.0
...,...,...,...,...,...,...,...
15521,15521,2c107d718e614961ae18c8ef2af37b03,offer completed,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,10.0
15532,15532,0b64be3b241c4407a5c9a71781173829,offer completed,0,NaN,ae264e3637204a6fb9bb56bc8210ddfd,10.0
15536,15536,1593d617fac246ef8e50dbb0ffd77f5f,offer completed,0,NaN,2906b810c7d4411798c6938adc9daaa5,2.0
15541,15541,f367a50b86d049799bbb0eb645ee834c,offer completed,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,10.0


---

In [19]:
# value 문자열 -> dict 변환
transcript['value'] = transcript['value'].apply(ast.literal_eval)

# offer_id 추출 ('offer id' 또는 'offer_id' 둘 다 대응)
transcript['offer_id'] = transcript['value'].apply(
    lambda x: x.get('offer id') if isinstance(x, dict) and 'offer id' in x
    else (x.get('offer_id') if isinstance(x, dict) and 'offer_id' in x
    else (x.get('amount') if isinstance(x, dict) and 'amount' in x else None))
)

df_tran = transcript.drop(columns='value')
# df_tran['offer_id'] = df_tran['value'].apply(
#     lambda x: x.get('offer id') if isinstance(x, dict) else None
# )

KeyError: 'value'

In [ ]:
df_tran1 = transcript.sort_values(['person', 'time'])
df_tran1.head(3)

In [ ]:
df_merge = df_tran.merge(
    profile[['offer_id', 'offer_type', 'reward', 'difficulty', 'duration']],
    on='offer_id',
    how='left'
)

df_merge.sort_values(['person', 'time']).head(1)

In [ ]:
first_event_time0 = (
    df_merge[df_merge['time'] == 0]
    .sort_values(['person', 'time'])
    .groupby('person')
    .first()
)

not_received_time0 = first_event_time0[first_event_time0['event'] != 'offer received']

print("time==0인데 received 아닌 사람 수:", len(not_received_time0))
display(not_received_time0[['event']].value_counts())

In [ ]:
# !jupyter nbconvert --to markdown "02_eda_v1.ipynb"